# Narkomfin Building Graph Analysis - PART 1

## 1. Import the needed libraries -

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [2]:
print(Helper.Version())

The version that you are using (0.9.50) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:

In [3]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 5. Load Type K floor plans (L1 and L2)

In [5]:
from pathlib import Path

HERE = Path.cwd()

# --- L1: single preprocessed Face ---
BREP_L1 = HERE.parent / '02_graph_analysis' / 'output' / 'L1_narkomfin_type_k_face.brep'
plan_l1 = Topology.ByBREPPath(str(BREP_L1))
print(f'L1 loaded — {Topology.TypeAsString(plan_l1)}')

# --- L2: raw cluster of 19 disconnected room faces ---
BREP_L2 = HERE.parent / '02_graph_analysis' / 'output' / 'L2_narkomfin_type_k.brep'
plan_l2_cluster = Topology.ByBREPPath(str(BREP_L2))
l2_room_faces   = Topology.Faces(plan_l2_cluster)
print(f'L2 loaded — {Topology.TypeAsString(plan_l2_cluster)}, {len(l2_room_faces)} room faces')

# Build a bounding face for L2 so the grid-slice pipeline works.
# Collect room boundary edges as extra cutters to preserve wall geometry.
all_verts = Topology.Vertices(plan_l2_cluster)
xs = [Vertex.X(v) for v in all_verts]
ys = [Vertex.Y(v) for v in all_verts]
z  = Vertex.Z(all_verts[0])

v1 = Vertex.ByCoordinates(min(xs), min(ys), z)
v2 = Vertex.ByCoordinates(max(xs), min(ys), z)
v3 = Vertex.ByCoordinates(max(xs), max(ys), z)
v4 = Vertex.ByCoordinates(min(xs), max(ys), z)
plan_l2 = Face.ByVertices([v1, v2, v3, v4])

# Room boundary edges — will be used as extra slicers in step 9
l2_room_edges = []
for f in l2_room_faces:
    l2_room_edges.extend(Topology.Edges(f))
l2_room_cutter = Cluster.ByTopologies(l2_room_edges)
print(f'L2 bounding face created — {len(l2_room_edges)} room boundary edges as cutters')

L1 loaded — Face
L2 loaded — Cluster, 19 room faces
L2 bounding face created — 428 room boundary edges as cutters


## 6. Show the geometry

In [6]:
# L1
Topology.Show(plan_l1,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='white',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

In [7]:
# L2 — show original room faces + bounding rectangle outline
Topology.Show(plan_l2_cluster, #plan_l2,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='white',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)

## 7. Create grid overlays

In [8]:
# --- L1 grid ---
b_r_l1   = Wire.BoundingRectangle(plan_l1)
d_l1     = Topology.Dictionary(b_r_l1)
xmin_l1  = Dictionary.ValueAtKey(d_l1, 'xmin')
xmax_l1  = Dictionary.ValueAtKey(d_l1, 'xmax')
ymin_l1  = Dictionary.ValueAtKey(d_l1, 'ymin')
ymax_l1  = Dictionary.ValueAtKey(d_l1, 'ymax')
width_l1 = Dictionary.ValueAtKey(d_l1, 'width')
length_l1= Dictionary.ValueAtKey(d_l1, 'length')
uRange_l1 = list(range(0, int(width_l1)+2, 2))
vRange_l1 = list(range(0, int(length_l1)+2, 2))
grid_l1  = Grid.EdgesByDistances(plan_l1, clip=True, uRange=uRange_l1, vRange=vRange_l1)
print(f'L1 grid: {len(uRange_l1)}x{len(vRange_l1)}')

# --- L2 grid (computed from vertex extents) ---
xmin_l2, xmax_l2 = min(xs), max(xs)
ymin_l2, ymax_l2 = min(ys), max(ys)
width_l2  = xmax_l2 - xmin_l2
length_l2 = ymax_l2 - ymin_l2
uRange_l2 = list(range(0, int(width_l2)+2, 2))
vRange_l2 = list(range(0, int(length_l2)+2, 2))
grid_l2  = Grid.EdgesByDistances(plan_l2, clip=True, uRange=uRange_l2, vRange=vRange_l2)
print(f'L2 grid: {len(uRange_l2)}x{len(vRange_l2)}')

L1 grid: 42x16
L2 grid: 42x16


## 8. Show the geometry and the grid

In [9]:
# L1 + grid
Topology.Show(plan_l1, grid_l1,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='grey',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800, height=500,
              renderer=renderer)

In [10]:
# L2 + grid
Topology.Show(plan_l2_cluster, grid_l2,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor='grey',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800, height=500,
              renderer=renderer)

## 9. Slice the floor plans with their grids to create topologic shells

In [19]:
# --- L1: slice single face by grid ---
shell_l1 = Topology.Slice(plan_l1, grid_l1)
faces_l1 = Topology.Faces(shell_l1)
for i, f in enumerate(faces_l1):
    d = Dictionary.ByKeyValue('face_id', 'l1_face_'+str(i+1))
    f = Topology.SetDictionary(f, d)
print(f'L1 shell: {len(faces_l1)} faces')

# --- L2: slice each room face individually by the grid, then combine ---
sliced_faces_l2 = []
for room_face in l2_room_faces:
    sliced = Topology.Slice(room_face, grid_l2)
    sliced_faces_l2.extend(Topology.Faces(sliced))

shell_l2 = Cluster.ByTopologies(sliced_faces_l2)
faces_l2 = sliced_faces_l2
for i, f in enumerate(faces_l2):
    d = Dictionary.ByKeyValue('face_id', 'l2_face_'+str(i+1))
    f = Topology.SetDictionary(f, d)
print(f'L2 result: {Topology.TypeAsString(shell_l2)}, {len(faces_l2)} faces')

L1 shell: 302 faces
L2 result: Cluster, 358 faces


## 10. Show the shell

In [20]:
# L1 shell
Topology.Show(shell_l1,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor='black',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800, height=500,
              renderer=renderer)

In [21]:
# L2 shell
Topology.Show(shell_l2,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor='black',
              edgeWidth=3,
              showVertices=False,
              backgroundColor='black',
              width=800, height=500,
              renderer=renderer)

## 11. Derive navigation and analysis graphs from the shell

In [22]:
# L1
navigation_graph_l1 = Graph.ByTopology(shell_l1, direct=False, viaSharedTopologies=True)
analysis_graph_l1   = Graph.ByTopology(shell_l1)

# L2
navigation_graph_l2 = Graph.ByTopology(shell_l2, direct=False, viaSharedTopologies=True)
analysis_graph_l2   = Graph.ByTopology(shell_l2)

print(f'L1 — analysis vertices: {len(Graph.Vertices(analysis_graph_l1))}, edges: {len(Graph.Edges(analysis_graph_l1))}')
print(f'L2 — analysis vertices: {len(Graph.Vertices(analysis_graph_l2))}, edges: {len(Graph.Edges(analysis_graph_l2))}')

L1 — analysis vertices: 302, edges: 475
L2 — analysis vertices: 358, edges: 0


## 12. Derive and store the analysis graph vertices

In [23]:
g_verts_l1 = Graph.Vertices(analysis_graph_l1)
g_verts_l2 = Graph.Vertices(analysis_graph_l2)

## 13. Show the analysis graph

In [24]:
# L1 analysis graph
Topology.Show(analysis_graph_l1,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor='red',
              edgeColor='lightgrey',
              backgroundColor='black',
              width=800, height=500,
              renderer=renderer)

In [25]:
# L2 analysis graph
Topology.Show(analysis_graph_l2,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor='red',
              edgeColor='lightgrey',
              backgroundColor='black',
              width=800, height=500,
              renderer=renderer)

## 14. Spatial Intelligence through Graph Analysis

### b. Shortest Path (Use navigation graph)

In [ ]:
import time

# --- L1 Shortest Path ---
start_l1 = Vertex.ByCoordinates(xmin_l1+2, ymax_l1-2, 0)
end_l1   = Vertex.ByCoordinates(xmax_l1-2, ymin_l1+2, 0)
crg_l1   = Graph.CompiledRoutingGraph(navigation_graph_l1, precomputeTurns=False)
t0 = time.time()
shortest_path_l1 = Graph.ShortestPath(crg_l1, vertexA=start_l1, vertexB=end_l1)
print('L1 — Shortest Path:', round(time.time()-t0, 2), 's')
straight_path_l1 = Wire.Straighten(shortest_path_l1, host=plan_l1)
print('  Original length:', round(Wire.Length(shortest_path_l1), 2))
print('  Straightened length:', round(Wire.Length(straight_path_l1), 2))
for edge in Topology.Edges(shortest_path_l1):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'red']))
for edge in Topology.Edges(straight_path_l1):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'blue']))

# --- L2 Shortest Path ---
start_l2 = Vertex.ByCoordinates(xmin_l2+2, ymax_l2-2, z)
end_l2   = Vertex.ByCoordinates(xmax_l2-2, ymin_l2+2, z)
crg_l2   = Graph.CompiledRoutingGraph(navigation_graph_l2, precomputeTurns=False)
t0 = time.time()
shortest_path_l2 = Graph.ShortestPath(crg_l2, vertexA=start_l2, vertexB=end_l2)
print('L2 — Shortest Path:', round(time.time()-t0, 2), 's')
straight_path_l2 = Wire.Straighten(shortest_path_l2, host=plan_l2)
print('  Original length:', round(Wire.Length(shortest_path_l2), 2))
print('  Straightened length:', round(Wire.Length(straight_path_l2), 2))
for edge in Topology.Edges(shortest_path_l2):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'red']))
for edge in Topology.Edges(straight_path_l2):
    edge = Topology.SetDictionary(edge, Dictionary.ByKeysValues(['width','color'], [7,'blue']))

In [ ]:
Topology.Show(plan_k, shortest_path_k, straight_path_k,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey='color',
              edgeWidthKey='width',
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


### c. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [ ]:
centrality_list_k = Graph.ClosenessCentrality(analysis_graph_k, colorScale='thermal')


* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell_k)
faces_k = Topology.Faces(shell_k)
_ = transfer_dicts_by_key(faces_k, g_verts_k, 'face_id')


In [ ]:
Topology.Show(faces_k,
              faceColorKey='cc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)


### d. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [ ]:
centrality_list_k = Graph.BetweennessCentrality(analysis_graph_k, normalize=True, colorScale='thermal')


* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell_k)
faces_k = Topology.Faces(shell_k)
_ = transfer_dicts_by_key(faces_k, g_verts_k, 'face_id')


In [ ]:
Topology.Show(faces_k,
              faceColorKey='bc_color',
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor='black',
              width=800,
              height=500,
              renderer=renderer)
